In [7]:
import pandas as pd
import numpy as np

# Dosya adını buraya yazıyoruz - Düzeltilmiş dosya yolu
# Ya çift ters eğik çizgi kullanın
dosya_yolu = 'C:\\Users\\EmreD\\Desktop\\Case_Study_2-20260120T031434Z-1-001\\Case_Study_2\\elektrik_veri_hashed.xlsx'

# Veya normal eğik çizgi kullanın (Python bunu otomatik olarak işleyebilir)
# dosya_yolu = 'C:/Users/EmreD/Desktop/Case_Study_2-20260120T031434Z-1-001/Case_Study_2/elektrik_veri_hashed.xlsx'

# Veya raw string kullanın
# dosya_yolu = r'C:\Users\EmreD\Desktop\Case_Study_2-20260120T031434Z-1-001\Case_Study_2\elektrik_veri_hashed.xlsx'

# Excel dosyasını okumak için açıyoruz
xls = pd.ExcelFile(dosya_yolu)

# Sayfaları tek tek değişkenlere atıyoruz
# Tahsilat sayfaları
df_tahsilat = pd.read_excel(xls, 'Tahsilat')
df_tahsilat_1 = pd.read_excel(xls, 'Tahsilat 1')

# Tüketim (Tahakkuk) sayfaları - Her ilçe ayrı sayfada
df_hamamozu = pd.read_excel(xls, 'Tahakkuk')
df_gumushacikoy = pd.read_excel(xls, 'Tahakkuk 1')
df_goynucek = pd.read_excel(xls, 'Tahakkuk 2')

# 3 ilçenin verisini tek bir tabloda birleştiriyoruz
df_tuketim = pd.concat([df_hamamozu, df_gumushacikoy, df_goynucek], ignore_index=True)

# Verilerin doğru yüklenip yüklenmediğini görmek için satır sayılarını yazdıralım
print("Tahsilat satır sayısı:", len(df_tahsilat))
print("Tahsilat 1 satır sayısı:", len(df_tahsilat_1))
print("Toplam Tüketim satır sayısı:", len(df_tuketim))

# Verinin ilk 5 satırına bakalım
print("\nBirleştirilmiş verinin ilk 5 satırı:")
print(df_tuketim.head())

Tahsilat satır sayısı: 636993
Tahsilat 1 satır sayısı: 917632
Toplam Tüketim satır sayısı: 1185698

Birleştirilmiş verinin ilk 5 satırı:
       il      ilce  sozlesme_hesap_no mali_yil_donem fatura_tarihi  \
0  AMASYA  HAMAMÖZÜ          917576806     2023-01-01    2023-01-12   
1  AMASYA  HAMAMÖZÜ          917576806     2023-01-01    2023-02-09   
2  AMASYA  HAMAMÖZÜ          917576806     2023-02-01    2023-02-09   
3  AMASYA  HAMAMÖZÜ          917576806     2023-02-01    2023-03-10   
4  AMASYA  HAMAMÖZÜ          917576806     2023-03-01    2023-03-10   

  kayit_tarihi vade_tarihi hesap_sinifi Hesap Sınıfı   kwh  
0   2023-03-06  2023-01-23         M001       Mesken  1.79  
1   2023-05-11  2023-02-20         M001       Mesken  2.60  
2   2023-05-11  2023-02-20         M001       Mesken  1.23  
3   2023-05-11  2023-03-20         M001       Mesken  2.56  
4   2023-05-11  2023-03-20         M001       Mesken  1.35  


In [8]:
# --- ADIM 2: VERİ TEMİZLEME VE BASİT İSTATİSTİKLER ---

# 1. Tarih sütununu düzeltiyoruz
# Excel'deki tarihi Python'ın anlayacağı formata çeviriyoruz
df_tuketim['tarih'] = pd.to_datetime(df_tuketim['mali_yil_donem'], errors='coerce')

# Analiz kolay olsun diye 'ay' bilgisini ayrı bir sütuna çıkarıyoruz
df_tuketim['ay'] = df_tuketim['tarih'].dt.month

# 2. Hatalı verileri (Negatif Tüketim) temizliyoruz
print("Temizlemeden önceki toplam kayıt:", len(df_tuketim))

# Sadece 0'dan büyük veya eşit olanları alıyoruz
df_tuketim = df_tuketim[df_tuketim['kwh'] >= 0]
print("Temizlendikten sonraki kayıt:", len(df_tuketim))

# 3. İlçe Karşılaştırması
# Her ilçede kaç müşteri var ve ortalama ne kadar elektrik tüketiyorlar?
ilce_tablosu = df_tuketim.groupby('ilce').agg({
    'sozlesme_hesap_no': 'nunique',  # Benzersiz müşteri sayısı
    'kwh': 'mean'                    # Ortalama tüketim
})

# Sütun isimlerini anlaşılır yapalım
ilce_tablosu.columns = ['Müşteri Sayısı', 'Ortalama Tüketim (kWh)']

print("\nİlçe Özet Tablosu:")
print(ilce_tablosu)

# 4. En çok tüketen abone grupları (Mesken mi, Sanayi mi?)
print("\nAbone türüne göre en çok tüketenler (Ortalama):")
print(df_tuketim.groupby('hesap_sinifi')['kwh'].mean().sort_values(ascending=False).head(5))

Temizlemeden önceki toplam kayıt: 1185698
Temizlendikten sonraki kayıt: 1185547

İlçe Özet Tablosu:
              Müşteri Sayısı  Ortalama Tüketim (kWh)
ilce                                                
GÖYNÜCEK                7128               89.736998
GÜMÜŞHACIKÖY           18190               97.473146
HAMAMÖZÜ                2981               70.892741

Abone türüne göre en çok tüketenler (Ortalama):
hesap_sinifi
A002    30203.431220
1       16594.174857
6       16155.254444
S001     7293.799626
T012     5213.509281
Name: kwh, dtype: float64


In [12]:
print("🔍 VERİ SAĞLIĞI KONTROLÜ:")

# 1. Eksik Veri Kontrolü
bos_degerler = df_tuketim.isnull().sum().sum()
print(f"   • Toplam Boş Hücre Sayısı: {bos_degerler}")

# 2. Negatif Değer Kontrolü (Hatalı/İade Fatura)
negatifler = df_tuketim[df_tuketim['kwh'] < 0]
print(f"   • Negatif (Hatalı) Tüketim Kaydı: {len(negatifler)}")

# 3. Sıfır Tüketim Kontrolü (Boş Evler)
sifir_tuketim = df_tuketim[df_tuketim['kwh'] == 0]
print(f"   • Sıfır Tüketim (Boş Ev/Abone): {len(sifir_tuketim)}")

# 4. Veri Tipleri
print("\n   • Sütun Veri Tipleri:")
print(df_tuketim.dtypes)

🔍 VERİ SAĞLIĞI KONTROLÜ:
   • Toplam Boş Hücre Sayısı: 0
   • Negatif (Hatalı) Tüketim Kaydı: 0
   • Sıfır Tüketim (Boş Ev/Abone): 55377

   • Sütun Veri Tipleri:
il                           object
ilce                         object
sozlesme_hesap_no             int64
mali_yil_donem               object
fatura_tarihi                object
kayit_tarihi                 object
vade_tarihi                  object
hesap_sinifi                 object
Hesap Sınıfı                 object
kwh                         float64
tarih                datetime64[ns]
ay                            int32
dtype: object


In [13]:
# 1. Tarih Dönüşümü (Mevsimsellik analizi için şart)
# 'mali_yil_donem' sütununu tarihe çeviriyoruz
df_tuketim['tarih'] = pd.to_datetime(df_tuketim['mali_yil_donem'], errors='coerce')
df_tuketim['yil'] = df_tuketim['tarih'].dt.year
df_tuketim['ay'] = df_tuketim['tarih'].dt.month

# 2. Negatifleri Temizleme
# Sadece 0 ve üzerindeki tüketimleri alarak 'df_temiz' oluşturuyoruz
df_temiz = df_tuketim[df_tuketim['kwh'] >= 0].copy()

print(f"🧹 Temizlik Tamamlandı.")
print(f"   • İşlem öncesi kayıt: {len(df_tuketim)}")
print(f"   • İşlem sonrası kayıt: {len(df_temiz)}")

🧹 Temizlik Tamamlandı.
   • İşlem öncesi kayıt: 1185547
   • İşlem sonrası kayıt: 1185547


In [14]:
# 1. İLÇE KARŞILAŞTIRMASI
# Hangi ilçe ne kadar tüketiyor?
ilce_ozet = df_temiz.groupby('ilce').agg({
    'sozlesme_hesap_no': 'nunique',      # Müşteri Sayısı
    'kwh': ['count', 'mean', 'sum']      # İşlem Hacmi, Ortalama, Toplam
})

ilce_ozet.columns = ['Müşteri Sayısı', 'İşlem Sayısı', 'Ort. Tüketim', 'Toplam Tüketim']
# Okunabilirlik için sıralama yapalım (Ortalamaya göre)
ilce_ozet = ilce_ozet.sort_values('Ort. Tüketim', ascending=False)

print("📊 İLÇE BAZLI İSTATİSTİKLER:")
print(ilce_ozet)

print("\n" + "-"*50 + "\n")

# 2. MÜŞTERİ TİPİ ANALİZİ (SEGMENTASYON)
# En çok elektrik tüketen müşteri grupları kimler?
segment_analizi = df_temiz.groupby('Hesap Sınıfı').agg({
    'kwh': ['mean', 'count']
})
segment_analizi.columns = ['Ortalama Tüketim', 'Kayıt Sayısı']
# En yüksek tüketime sahip ilk 5 grubu getir
print("🏭 EN YÜKSEK TÜKETİMLİ MÜŞTERİ GRUPLARI (TOP 5):")
print(segment_analizi.sort_values('Ortalama Tüketim', ascending=False).head(5))

📊 İLÇE BAZLI İSTATİSTİKLER:
              Müşteri Sayısı  İşlem Sayısı  Ort. Tüketim  Toplam Tüketim
ilce                                                                    
GÜMÜŞHACIKÖY           18190        765550     97.473146     74620566.74
GÖYNÜCEK                7128        295183     89.736998     26488836.34
HAMAMÖZÜ                2981        124814     70.892741      8848406.62

--------------------------------------------------

🏭 EN YÜKSEK TÜKETİMLİ MÜŞTERİ GRUPLARI (TOP 5):
                                        Ortalama Tüketim  Kayıt Sayısı
Hesap Sınıfı                                                          
Karayolları Genel Müdürlüğü Aydınlatma      30203.431220            41
Aritma Tesisleri                            16594.174857            35
Lisansız Üreticiler                         16155.254444           180
Sanayi                                       7293.799626           187
İçme-Kullanma Suyu (Belediye)                5213.509281           417


In [15]:
# IQR Yöntemi ile Aykırı Değer Hesaplama
Q1 = df_temiz['kwh'].quantile(0.25)
Q3 = df_temiz['kwh'].quantile(0.75)
IQR = Q3 - Q1

alt_sinir = Q1 - 1.5 * IQR
ust_sinir = Q3 + 1.5 * IQR

outliers = df_temiz[(df_temiz['kwh'] > ust_sinir)]

print("📈 AYKIRI DEĞER ANALİZİ (IQR):")
print(f"   • Normal Tüketim Aralığı: {alt_sinir:.2f} kWh ile {ust_sinir:.2f} kWh arası")
print(f"   • Aykırı (Aşırı Yüksek) Kayıt Sayısı: {len(outliers)}")
print(f"   • Toplam Veriye Oranı: %{(len(outliers)/len(df_temiz)*100):.2f}")

📈 AYKIRI DEĞER ANALİZİ (IQR):
   • Normal Tüketim Aralığı: -74.95 kWh ile 172.97 kWh arası
   • Aykırı (Aşırı Yüksek) Kayıt Sayısı: 48454
   • Toplam Veriye Oranı: %4.09


In [16]:
# Temiz veriyi CSV olarak kaydet (Hızlı okuma için)
df_temiz.to_csv('temiz_tuketim_verisi.csv', index=False)

# Tahsilat 1 verisini de ödeme analizi için kaydedelim
df_tahsilat_1.to_csv('temiz_tahsilat_verisi.csv', index=False)

print("💾 DOSYALAR BAŞARIYLA KAYDEDİLDİ:")
print("   -> temiz_tuketim_verisi.csv")
print("   -> temiz_tahsilat_verisi.csv")
print("\n✅ Notebook 1 tamamlandı! Şimdi 'notebook_02_gorsellestirme'ye geçebilirsin.")

💾 DOSYALAR BAŞARIYLA KAYDEDİLDİ:
   -> temiz_tuketim_verisi.csv
   -> temiz_tahsilat_verisi.csv

✅ Notebook 1 tamamlandı! Şimdi 'notebook_02_gorsellestirme'ye geçebilirsin.
